In [12]:
import os
os.chdir("/home/ubuntu/work/dahyeon/backend")
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [13]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.modules.RAG.retriever import (
    full_bm25,
    full_dense,
    full_hybrid,
    chunk_dense
)
import sys
from app.modules.RAG.anchor_book_pipeline import run_anchor_pipeline
from app.core.config import settings
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
nest_asyncio.apply()

In [14]:
REPO_ROOT     = Path("/home/ubuntu/work/dahyeon")
DATASET_DIR   = REPO_ROOT / "evaluation" / "dataset"
SCENARIO_PATH = DATASET_DIR / "scenario_data.json"

with open(SCENARIO_PATH, encoding="utf-8") as f:
    scenarios = json.load(f)

def extract_rag_query(scenario: dict) -> dict:
    """마지막 turn의 rag_query를 꺼내고 query_id(=scenario_id)를 추가한다."""
    rag_query = scenario["turns"][-1]["rag_query"].copy()
    rag_query["query_id"] = scenario["scenario_id"]
    return rag_query

In [15]:
def simplify_item(item, source_name):
    return {
        "isbn":         item.get("isbn"),
        "title":        item.get("title"),
        "author":       item.get("author"),
        "publisher":    item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page":         item.get("page"),
        "large_cate":   item.get("large_cate"),
        "mid_cate":     item.get("mid_cate"),
        "small_cate":   item.get("small_cate"),
        "book_intro":   item.get("book_intro"),
        "book_index":   item.get("book_index"),
        "review_count": item.get("review_count"),
        "review_text":  item.get("review_text"),
    }


In [16]:
# ============================================================
# Strategy0 Hybrid BM25/Dense 가중치 실험
# - index: books_review_full
# - retrieval: full_hybrid
# - embedding model: KURE
# - bm25_weight / dense_weight 조절
# ============================================================

from pathlib import Path
import json
import pandas as pd
from tqdm import tqdm

INDEX_NAME = "books_review_full"

EMBEDDING_MODEL = "kure"
EMBEDDING_FIELD = "embedding_kure"

TOP_K = 20
BM25_CANDIDATE_SIZE = 100
DENSE_CANDIDATE_SIZE = 100
NUM_CANDIDATES = 300

WEIGHT_CONFIGS = [
    {"bm25_weight": 0.9, "dense_weight": 0.1},
    {"bm25_weight": 0.8, "dense_weight": 0.2},
    {"bm25_weight": 0.7, "dense_weight": 0.3},
    {"bm25_weight": 0.6, "dense_weight": 0.4},
    {"bm25_weight": 0.5, "dense_weight": 0.5},
    {"bm25_weight": 0.4, "dense_weight": 0.6},
    {"bm25_weight": 0.3, "dense_weight": 0.7},
    {"bm25_weight": 0.2, "dense_weight": 0.8},
    {"bm25_weight": 0.1, "dense_weight": 0.9},
]

OUTPUT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/strategy0_hybrid_weight_eval.jsonl")

all_candidates = []

for scenario in tqdm(scenarios, desc="strategy0 hybrid weight experiment"):

    sample_result = extract_rag_query(scenario)

    if sample_result.get("anchor"):
        sample_result = run_anchor_pipeline(sample_result)

    query_id = sample_result["query_id"]

    for weight_id, weights in enumerate(WEIGHT_CONFIGS, start=1):

        bm25_weight = weights["bm25_weight"]
        dense_weight = weights["dense_weight"]

        re_hybrid = full_hybrid(
            result=sample_result,
            index_name=INDEX_NAME,
            size=TOP_K,
            bm25_candidate_size=BM25_CANDIDATE_SIZE,
            dense_candidate_size=DENSE_CANDIDATE_SIZE,
            num_candidates=NUM_CANDIDATES,
            require_both=False,
            overlap_bonus=0.0,
            rrf_k=60,
            bm25_weight=bm25_weight,
            dense_weight=dense_weight,
            embedding_field=EMBEDDING_FIELD,
            embedding_model=EMBEDDING_MODEL
        )

        for rank, book in enumerate(re_hybrid, start=1):
            all_candidates.append({
                "query_id": query_id,

                "strategy": "strategy0",
                "retrieval_type": "full_hybrid",
                "index_name": INDEX_NAME,

                "weight_id": weight_id,
                "bm25_weight": bm25_weight,
                "dense_weight": dense_weight,

                "rank": rank,
                "score": book.get("score"),

                "bm25_rank": book.get("bm25_rank"),
                "dense_rank": book.get("dense_rank"),
                "bm25_raw_score": book.get("bm25_raw_score"),
                "dense_raw_score": book.get("dense_raw_score"),
                "bm25_rrf_score": book.get("bm25_rrf_score"),
                "dense_rrf_score": book.get("dense_rrf_score"),
                "has_bm25": book.get("has_bm25"),
                "has_dense": book.get("has_dense"),

                "isbn": str(book.get("isbn")),
                "title": book.get("title"),
                "author": book.get("author"),
                "publisher": book.get("publisher"),
                "publish_date": book.get("publish_date"),
                "page": book.get("page"),

                "large_cate": book.get("large_cate") or book.get("large"),
                "mid_cate": book.get("mid_cate"),
                "small_cate": book.get("small_cate"),

                "book_intro": book.get("book_intro"),
                "book_index": book.get("book_index"),
                "book_index_text": book.get("book_index_text"),
                "review": book.get("review"),
                "review_text": book.get("review_text"),
            })

print(f"\n전체 후보 도서: {len(all_candidates):,}")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for item in all_candidates:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"저장 완료: {OUTPUT_PATH}")

strategy0_weight_df = pd.DataFrame(all_candidates)
strategy0_weight_df.head()

strategy0 hybrid weight experiment: 100%|██████████| 21/21 [05:52<00:00, 16.77s/it]


전체 후보 도서: 3,780
저장 완료: /home/ubuntu/work/dahyeon/evaluation/dataset/strategy0_hybrid_weight_eval.jsonl


,query_id,strategy,retrieval_type,index_name,weight_id,bm25_weight,dense_weight,rank,score,bm25_rank,...,publish_date,page,large_cate,mid_cate,small_cate,book_intro,book_index,book_index_text,review,review_text
0,S01,strategy0,full_hybrid,books_review_full,1,0.9,0.1,1,0.016225,1.0,...,2022-09-16,288,[인문],[인문학일반],[인문교양],"『물고기와 철학자』, 『바다로 떠난 허수아비』에 이은\n삶을 사유하는 철학자 임판 ...",,시작하는 말 / 트램펄린 위의 사람들 트램펄린에서 살아간다면 흔들림 속에서 떠오르는...,"{'recommended_for': '삶의 의미를 고민하는 모든 분들', 'stre...",실재나 참된 세상이 우리 세상의 그림자에 불과함을 깨닫는 순간 남는 것은 허무가 아...
1,S01,strategy0,full_hybrid,books_review_full,1,0.9,0.1,2,0.015396,4.0,...,2025-06-20,256,"[시/에세이, 인문]","[나라별 에세이, 철학, 테마에세이]","[교양철학, 노년에세이, 한국에세이]","영원히 살고자 하는 욕망이 부른 인류세 시대,\n이제 우리는 나이 듦의 윤리를 말해...",,수명연장 과학 시대의 나이 듦과 삶의 의미 김종갑 노화가 질병이라는 말에 관하여 서...,None,
2,S01,strategy0,full_hybrid,books_review_full,1,0.9,0.1,3,0.015373,3.0,...,2024-07-03,256,[인문],[철학],[철학에세이],현실을 바꿔나갈 힘을 얻는 ‘현장의 인문학’\n이 책의 제목에 나오는 ‘하녀’는 권...,,철학자와 하녀 그리고 별에 관한 이야기 철학은 지옥에서 하는 것이다 배움 이전에 배...,"{'recommended_for': '', 'strengths': '생활 속에서 철...","철학책을 재미있게 읽을 수 있으며, 삶의 방식과 철학적 사고를 연결해주는 유익한 경..."
3,S01,strategy0,full_hybrid,books_review_full,1,0.9,0.1,4,0.014875,8.0,...,2025-03-10,232,"[시/에세이, 인문]",[나라별 에세이],[한국에세이],"“인간존재와 삶의 의미는 무엇인지, 왜 사는지, 무엇을 위해 어떻게 살 것인지, 늙...",,,"{'recommended_for': '삶의 본질을 알고자 하거나 힘든 분들', 's...",속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드는 책입니다. 삶의 본질을 알고자 하거...
4,S01,strategy0,full_hybrid,books_review_full,1,0.9,0.1,5,0.014747,6.0,...,2020-02-05,228,[시/에세이],[나라별 에세이],[한국에세이],생의 마지막에서 갈구하는 건 소소한 행복이었다!\n우리는 살아가면서 “지금 나는 잘...,,,"{'recommended_for': '', 'strengths': '마음을 따뜻하게...","삶의 시작만큼이나 끝도 중요하다는 것을 깨닫고, 더 많이 사랑해야겠다는 다짐을 하게..."


In [17]:
from pathlib import Path
import sys
import json
import pandas as pd

# =====================================================
# 1. 파일 경로
# =====================================================

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final_corrected.json")
WEIGHT_RESULT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/strategy0_hybrid_weight_eval.jsonl")

REPO_ROOT = Path("/home/ubuntu/work/dahyeon")
sys.path.insert(0, str(REPO_ROOT / "evaluation" / "metrics"))

from metrics import hit_at_k, recall_at_k, mrr_at_k, ndcg_at_k

K_VALUES = [20]
RELEVANCE_THRESHOLD = 2

# =====================================================
# 2. goldset 로드
# =====================================================

with GOLDSET_PATH.open("r", encoding="utf-8") as f:
    goldset = json.load(f)

gold_df = pd.DataFrame(goldset)

relevant_by_query = {}

for qid, g in gold_df.groupby("query_id"):
    relevant_isbns = (
        g[g["final_grade"] >= RELEVANCE_THRESHOLD]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )
    relevant_by_query[qid] = set(relevant_isbns)

print("query 수:", len(relevant_by_query))
print("relevant 있는 query 수:", sum(1 for v in relevant_by_query.values() if v))

# =====================================================
# 3. 가중치 실험 결과 로드
# =====================================================

rows = []

with WEIGHT_RESULT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

weight_df = pd.DataFrame(rows)

print("가중치 실험 결과 개수:", len(weight_df))
display(weight_df.head())

# =====================================================
# 4. eval_data 구성
#    기준: query_id + weight_id
# =====================================================

eval_data = {}

for qid in relevant_by_query:
    eval_data[qid] = {
        "relevant_isbns": relevant_by_query[qid]
    }

for (qid, weight_id), g in weight_df.groupby(["query_id", "weight_id"]):

    g = g.sort_values("rank")

    ranked = (
        g["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    if qid not in eval_data:
        eval_data[qid] = {
            "relevant_isbns": set()
        }

    eval_data[qid][weight_id] = ranked

weight_ids = sorted(weight_df["weight_id"].dropna().unique())

print("weight_ids:", weight_ids)

# =====================================================
# 5. weight_id별 retrieval metric 계산
# =====================================================

def compute_weight_metrics(eval_data, weight_ids, ks=K_VALUES):
    metric_rows = []

    for qid, d in eval_data.items():
        rel = d["relevant_isbns"]

        if not rel:
            continue

        for weight_id in weight_ids:
            ranked = d.get(weight_id, [])

            if not ranked:
                continue

            for k in ks:
                topk = ranked[:k]
                hit_isbns = set(topk) & set(rel)

                metric_rows.append({
                    "query_id": qid,
                    "weight_id": weight_id,
                    "k": k,

                    "hit": hit_at_k(rel, ranked, k),
                    "recall": recall_at_k(rel, ranked, k),
                    "mrr": mrr_at_k(rel, ranked, k),
                    "ndcg": ndcg_at_k(rel, ranked, k),

                    "goldset_count": len(rel),
                    "retrieved_relevant_count": len(hit_isbns),
                    "matched_isbns": list(hit_isbns),
                })

    return pd.DataFrame(metric_rows)

retrieval_df = compute_weight_metrics(
    eval_data=eval_data,
    weight_ids=weight_ids,
    ks=K_VALUES
)

print(f"총 {len(retrieval_df)}개 행 (시나리오 × weight_id × K)")
display(retrieval_df.head())

# =====================================================
# 6. weight 정보 붙이기
# =====================================================

weight_info_df = (
    weight_df[["weight_id", "bm25_weight", "dense_weight"]]
    .drop_duplicates()
    .sort_values("weight_id")
)

retrieval_df = retrieval_df.merge(
    weight_info_df,
    on="weight_id",
    how="left"
)

# =====================================================
# 7. weight별 평균 recall@20 요약
# =====================================================

summary_df = (
    retrieval_df
    .groupby(["weight_id", "bm25_weight", "dense_weight"])
    .agg({
        "hit": "mean",
        "recall": "mean",
        "mrr": "mean",
        "ndcg": "mean",
        "retrieved_relevant_count": "sum",
        "goldset_count": "sum",
    })
    .reset_index()
    .sort_values("recall", ascending=False)
)

display(summary_df)

query 수: 21
relevant 있는 query 수: 21
가중치 실험 결과 개수: 3780


,query_id,strategy,retrieval_type,index_name,weight_id,bm25_weight,dense_weight,rank,score,bm25_rank,...,publish_date,page,large_cate,mid_cate,small_cate,book_intro,book_index,book_index_text,review,review_text
0,S01,strategy0,full_hybrid,books_review_full,1,0.9,0.1,1,0.016225,1.0,...,2022-09-16,288,[인문],[인문학일반],[인문교양],"『물고기와 철학자』, 『바다로 떠난 허수아비』에 이은\n삶을 사유하는 철학자 임판 ...",,시작하는 말 / 트램펄린 위의 사람들 트램펄린에서 살아간다면 흔들림 속에서 떠오르는...,"{'recommended_for': '삶의 의미를 고민하는 모든 분들', 'stre...",실재나 참된 세상이 우리 세상의 그림자에 불과함을 깨닫는 순간 남는 것은 허무가 아...
1,S01,strategy0,full_hybrid,books_review_full,1,0.9,0.1,2,0.015396,4.0,...,2025-06-20,256,"[시/에세이, 인문]","[나라별 에세이, 철학, 테마에세이]","[교양철학, 노년에세이, 한국에세이]","영원히 살고자 하는 욕망이 부른 인류세 시대,\n이제 우리는 나이 듦의 윤리를 말해...",,수명연장 과학 시대의 나이 듦과 삶의 의미 김종갑 노화가 질병이라는 말에 관하여 서...,None,
2,S01,strategy0,full_hybrid,books_review_full,1,0.9,0.1,3,0.015373,3.0,...,2024-07-03,256,[인문],[철학],[철학에세이],현실을 바꿔나갈 힘을 얻는 ‘현장의 인문학’\n이 책의 제목에 나오는 ‘하녀’는 권...,,철학자와 하녀 그리고 별에 관한 이야기 철학은 지옥에서 하는 것이다 배움 이전에 배...,"{'recommended_for': '', 'strengths': '생활 속에서 철...","철학책을 재미있게 읽을 수 있으며, 삶의 방식과 철학적 사고를 연결해주는 유익한 경..."
3,S01,strategy0,full_hybrid,books_review_full,1,0.9,0.1,4,0.014875,8.0,...,2025-03-10,232,"[시/에세이, 인문]",[나라별 에세이],[한국에세이],"“인간존재와 삶의 의미는 무엇인지, 왜 사는지, 무엇을 위해 어떻게 살 것인지, 늙...",,,"{'recommended_for': '삶의 본질을 알고자 하거나 힘든 분들', 's...",속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드는 책입니다. 삶의 본질을 알고자 하거...
4,S01,strategy0,full_hybrid,books_review_full,1,0.9,0.1,5,0.014747,6.0,...,2020-02-05,228,[시/에세이],[나라별 에세이],[한국에세이],생의 마지막에서 갈구하는 건 소소한 행복이었다!\n우리는 살아가면서 “지금 나는 잘...,,,"{'recommended_for': '', 'strengths': '마음을 따뜻하게...","삶의 시작만큼이나 끝도 중요하다는 것을 깨닫고, 더 많이 사랑해야겠다는 다짐을 하게..."


weight_ids: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
총 189개 행 (시나리오 × weight_id × K)


,query_id,weight_id,k,hit,recall,mrr,ndcg,goldset_count,retrieved_relevant_count,matched_isbns
0,S01,1,20,1,0.411765,1.0,0.510848,17,7,"[9791196539757, 9791189088361, 9791157063598, ..."
1,S01,2,20,1,0.411765,1.0,0.514884,17,7,"[9791196539757, 9791189088361, 9791157063598, ..."
2,S01,3,20,1,0.411765,1.0,0.528390,17,7,"[9791196539757, 9791189088361, 9791157063598, ..."
3,S01,4,20,1,0.470588,1.0,0.564296,17,8,"[9791196539757, 9791194741190, 9791157063598, ..."
4,S01,5,20,1,0.470588,1.0,0.555862,17,8,"[9791141023201, 9791196539757, 9791194741190, ..."


,weight_id,bm25_weight,dense_weight,hit,recall,mrr,ndcg,retrieved_relevant_count,goldset_count
6,7,0.3,0.7,0.952381,0.350625,0.752834,0.494000,165,580
8,9,0.1,0.9,0.952381,0.331514,0.807143,0.487541,159,580
5,6,0.4,0.6,0.952381,0.329078,0.736508,0.474015,156,580
7,8,0.2,0.8,0.952381,0.328496,0.792517,0.489836,160,580
4,5,0.5,0.5,0.952381,0.296775,0.765306,0.446567,141,580
3,4,0.6,0.4,0.952381,0.281048,0.742659,0.420179,128,580
2,3,0.7,0.3,0.952381,0.269958,0.759921,0.403118,121,580
1,2,0.8,0.2,0.952381,0.264808,0.740741,0.389994,119,580
0,1,0.9,0.1,0.904762,0.247624,0.653187,0.358166,110,580


In [18]:
# =====================================================
# 1. retrieval 실행
# =====================================================

TOP_K = 20

all_results = []

for scenario in tqdm(scenarios, desc="retrieval evaluation"):

    sample_result = extract_rag_query(scenario)

    if sample_result.get("anchor"):
        sample_result = run_anchor_pipeline(sample_result)

    query_id = sample_result["query_id"]

    # -------------------------------------------------
    # BM25
    # -------------------------------------------------

    bm25_results = full_bm25(
        result=sample_result,
        index_name="books_review_full",
        size=TOP_K,
    )

    # -------------------------------------------------
    # Dense
    # -------------------------------------------------

    dense_results = full_dense(
        result=sample_result,
        index_name="books_review_full",
        size=TOP_K,
        embedding_field="embedding_kure",
        embedding_model="kure"
    )

    # -------------------------------------------------
    # Hybrid
    # -------------------------------------------------

    hybrid_results = full_hybrid(
        result=sample_result,
        index_name="books_review_full",

        size=TOP_K,

        bm25_weight=0.3,
        dense_weight=0.7,

        embedding_field="embedding_kure",
        embedding_model="kure"
    )

    retrieval_map = {
        "bm25": bm25_results,
        "dense": dense_results,
        "hybrid": hybrid_results,
    }

    for retrieval_name, results in retrieval_map.items():

        seen = set()

        for rank, book in enumerate(results, start=1):

            isbn = str(book["isbn"])

            if isbn in seen:
                continue

            seen.add(isbn)

            all_results.append({
                "query_id": query_id,
                "retriever": retrieval_name,
                "rank": rank,
                "isbn": isbn,
                "score": book.get("score"),
            })

# =====================================================
# 2. dataframe
# =====================================================

result_df = pd.DataFrame(all_results)

display(result_df.head())

retrieval evaluation: 100%|██████████| 21/21 [01:57<00:00,  5.59s/it]


,query_id,retriever,rank,isbn,score
0,S01,bm25,1,9791157833733,60.063198
1,S01,bm25,2,9791193358566,49.496807
2,S01,bm25,3,9791189088361,46.529346
3,S01,bm25,4,9791165347772,44.275227
4,S01,bm25,5,9791157063598,38.949726


In [22]:
GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final_corrected.json")

with GOLDSET_PATH.open("r", encoding="utf-8") as f:
    goldset_data = json.load(f)

goldset_df = pd.DataFrame(goldset_data)

# =====================================================
# 3. eval_data 구성
# =====================================================

# goldset_df에 query_id, isbn 컬럼이 있다고 가정
# relevance_grade가 2, 3인 것만 relevant로 사용
relevant_by_query = (
    goldset_df[goldset_df["relevance_grade"].isin([2, 3])]
    .groupby("query_id")["isbn"]
    .apply(lambda x: x.dropna().astype(str).unique().tolist())
    .to_dict()
)

eval_data = {
    qid: {
        "relevant_isbns": rel_isbns
    }
    for qid, rel_isbns in relevant_by_query.items()
}

for (qid, retriever), g in result_df.groupby(["query_id", "retriever"]):

    if qid not in eval_data:
        continue

    ranked = (
        g.sort_values("rank")["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    eval_data[qid][retriever] = ranked

retrievers = ["bm25", "dense", "hybrid"]

def compute_retrieval_metrics(
    eval_data,
    retrievers,
    ks=[20],
):
    rows = []

    for qid, d in eval_data.items():
        rel = d["relevant_isbns"]

        if not rel:
            continue

        for retriever in retrievers:
            ranked = d.get(retriever, [])

            if not ranked:
                continue

            for k in ks:
                topk = ranked[:k]
                hit_isbns = set(topk) & set(rel)

                rows.append({
                    "query_id": qid,
                    "retriever": retriever,
                    "k": k,

                    "hit": hit_at_k(rel, ranked, k),
                    "recall": recall_at_k(rel, ranked, k),
                    "mrr": mrr_at_k(rel, ranked, k),
                    "ndcg": ndcg_at_k(rel, ranked, k),

                    "goldset_count": len(rel),
                    "retrieved_relevant_count": len(hit_isbns),
                    "matched_isbns": list(hit_isbns),
                })

    return pd.DataFrame(rows)

retrieval_df = compute_retrieval_metrics(
    eval_data=eval_data,
    retrievers=retrievers,
    ks=[20]
)

summary_df = (
    retrieval_df
    .groupby("retriever")
    .agg({
        "hit": "mean",
        "recall": "mean",
        "mrr": "mean",
        "ndcg": "mean",
        "retrieved_relevant_count": "sum",
        "goldset_count": "sum",
    })
    .reset_index()
    .sort_values("recall", ascending=False)
)

display(summary_df)

,retriever,hit,recall,mrr,ndcg,retrieved_relevant_count,goldset_count
2,hybrid,0.952381,0.350784,0.800454,0.526552,181,666
1,dense,0.952381,0.332025,0.753968,0.510137,175,666
0,bm25,0.904762,0.235542,0.525718,0.350253,121,666


In [23]:
retrieval_df.to_csv(
    "/home/ubuntu/work/dahyeon/evaluation/dataset/retrieval_3_metric_results.csv",
    index=False,
    encoding="utf-8-sig"
)

In [25]:
import json
import pandas as pd
from pathlib import Path

# =====================================================
# 1. 파일 경로
# =====================================================

SCENARIO_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/scenario_data_after_anchor_rewrite.json")
SAVE_DIR = Path("/home/ubuntu/work/dahyeon/evaluation/dataset")

# retrieval_df는 이미 위에서 계산된 상태라고 가정
metric_df = retrieval_df.copy()

# =====================================================
# 2. scenario 데이터 로드
# =====================================================

with SCENARIO_PATH.open("r", encoding="utf-8") as f:
    scenario_data = json.load(f)

# =====================================================
# 3. query_type 추가
# =====================================================

query_type_map = {
    item["scenario_id"]: item["query_type"]
    for item in scenario_data
}

metric_df["query_type"] = metric_df["query_id"].map(query_type_map)

# =====================================================
# 4. retrieval query 추가
# =====================================================

query_text_map = {}

for item in scenario_data:
    qid = item["scenario_id"]
    query = None

    for turn in reversed(item.get("turns", [])):
        if "rag_query" in turn:
            query = turn["rag_query"].get("semantic_query")
            break

    if query is None and item.get("turns"):
        query = item["turns"][-1].get("user_input")

    query_text_map[qid] = query

metric_df["query"] = metric_df["query_id"].map(query_text_map)

# =====================================================
# 5. 전체 평균 성능
# =====================================================

overall_result = (
    metric_df
    .groupby("retriever")[["hit", "recall", "mrr", "ndcg"]]
    .mean()
    .round(4)
    .reset_index()
    .sort_values("recall", ascending=False)
)

print("=== 전체 평균 성능 ===")
print(overall_result)

# =====================================================
# 6. query_type별 성능
# =====================================================

type_result = (
    metric_df
    .groupby(["query_type", "retriever"])[["hit", "recall", "mrr", "ndcg"]]
    .mean()
    .round(4)
    .reset_index()
)

print("\n=== Query Type별 성능 ===")
print(type_result)

# =====================================================
# 7. query_type별 recall 비교
# =====================================================

type_recall_pivot = type_result.pivot(
    index="query_type",
    columns="retriever",
    values="recall"
)

print("\n=== Query Type별 Recall 비교 ===")
print(type_recall_pivot)

# =====================================================
# 8. query별 recall 비교
# =====================================================

query_recall_pivot = metric_df.pivot_table(
    index="query_id",
    columns="retriever",
    values="recall"
).reset_index()

query_recall_pivot["query_type"] = query_recall_pivot["query_id"].map(query_type_map)
query_recall_pivot["query"] = query_recall_pivot["query_id"].map(query_text_map)

print("\n=== Query별 Recall 비교 ===")
print(query_recall_pivot)

# =====================================================
# 9. query별 최고 retriever
# =====================================================

retriever_cols = [
    c for c in query_recall_pivot.columns
    if c not in ["query_id", "query_type", "query"]
]

query_recall_pivot["best_retriever"] = (
    query_recall_pivot[retriever_cols]
    .idxmax(axis=1)
)

query_recall_pivot["best_recall"] = (
    query_recall_pivot[retriever_cols]
    .max(axis=1)
)

print("\n=== Query별 최고 Retriever ===")
print(
    query_recall_pivot[
        ["query_id", "query_type", "query"] 
        + retriever_cols 
        + ["best_retriever", "best_recall"]
    ]
)

# =====================================================
# 10. 결과 저장
# =====================================================

# metric_df.to_csv(
#     SAVE_DIR / "retriever_metric_with_query_type.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# overall_result.to_csv(
#     SAVE_DIR / "retriever_overall_result.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# type_result.to_csv(
#     SAVE_DIR / "retriever_query_type_result.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# type_recall_pivot.to_csv(
#     SAVE_DIR / "retriever_query_type_recall_compare.csv",
#     encoding="utf-8-sig"
# )

# query_recall_pivot.to_csv(
#     SAVE_DIR / "retriever_query_recall_compare.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# print("\n저장 완료")

=== 전체 평균 성능 ===
  retriever     hit  recall     mrr    ndcg
2    hybrid  0.9524  0.3508  0.8005  0.5266
1     dense  0.9524  0.3320  0.7540  0.5101
0      bm25  0.9048  0.2355  0.5257  0.3503

=== Query Type별 성능 ===
            query_type retriever     hit  recall     mrr    ndcg
0         anchor_based      bm25  0.6667  0.3377  0.4444  0.2570
1         anchor_based     dense  0.6667  0.4211  0.6667  0.4187
2         anchor_based    hybrid  0.6667  0.4912  0.6667  0.4738
3   availability_first      bm25  1.0000  0.2292  0.7143  0.5753
4   availability_first     dense  1.0000  0.2544  0.7778  0.6230
5   availability_first    hybrid  1.0000  0.2856  1.0000  0.7084
6      broad_ambiguous      bm25  1.0000  0.1082  0.7083  0.3537
7      broad_ambiguous     dense  1.0000  0.2283  0.7778  0.5451
8      broad_ambiguous    hybrid  1.0000  0.2176  0.7778  0.5227
9      pure_mood_state      bm25  1.0000  0.1537  0.2019  0.1268
10     pure_mood_state     dense  1.0000  0.3790  0.7500  0.4010
11 